# Setup

In [ ]:
import duckdb
from splink import Linker, DuckDBAPI, SettingsCreator, block_on
import splink.comparison_library as cl
import splink.comparison_level_library as cll

In [ ]:
con = duckdb.connect()

df_l = con.read_parquet("./perturbed_data_l.parquet")
df_r = con.read_parquet("./perturbed_data_r.parquet")

db_api = DuckDBAPI(con)

df_l = db_api.register(df_l, dataset_display_name="df_l")
df_r = db_api.register(df_r, dataset_display_name="df_r")

# Defining comparisons and estimating parameters

In [ ]:
# get dataset count
N_l = con.sql(f"select count(*) as n from {df_l.physical_name}").to_dicts()[0]['n']

In [ ]:
geography_comparison = cl.CustomComparison(
    comparison_levels=[
        cll.And(cll.NullLevel("address"), cll.NullLevel("sa4")),
        cll.ArrayIntersectLevel("address", 1),
        cll.ArrayIntersectLevel("sa4", 1),
        cll.ElseLevel(),
    ],
    output_column_name="geography",
)

dob_comparison = cl.CustomComparison(
    comparison_levels=[
        cll.NullLevel("dob"),
        cll.ExactMatchLevel("dob"),
        cll.LevenshteinLevel("cast(dob as string)", 1),
        cll.AbsoluteDateDifferenceLevel("dob", input_is_string=False, threshold=1, metric="month"),
        cll.AbsoluteDateDifferenceLevel("dob", input_is_string=False, threshold=1, metric="year"),
        cll.ElseLevel(),
    ],
    output_column_name="dob",
)

settings = SettingsCreator(
    link_type="link_and_dedupe",
    comparisons=[
        cl.JaroWinklerAtThresholds("fname", [0.9]),
        cl.JaroWinklerAtThresholds("sname", [0.9]),
        cl.ExactMatch("sex"),
        dob_comparison,
        geography_comparison,
    ],
    blocking_rules_to_generate_predictions=[
        block_on("fname", "sname"),
        block_on("dob"),
        block_on("address", arrays_to_explode=["address"]),
    ],
    additional_columns_to_retain=["person_id", "family_id"],
    retain_intermediate_calculation_columns=True,
    retain_matching_columns=True,
    probability_two_random_records_match=1/N_l,
)

linker = Linker(
    splink_dataframe_or_dataframes=[df_l, df_r],
    settings=settings,
    log_level=10,
)

In [ ]:
linker.training.estimate_u_using_random_sampling(
    max_pairs=1e7,
    num_chunks=1,
    min_count_per_level=None,
)

In [ ]:
ts1 = linker.training.estimate_parameters_using_expectation_maximisation(
    blocking_rule=block_on("dob"),
    estimate_without_term_frequencies=True,
)

ts2 = linker.training.estimate_parameters_using_expectation_maximisation(
    blocking_rule=block_on("fname", "substring(sname, 1, 2)"),
    estimate_without_term_frequencies=True,
)

ts3 = linker.training.estimate_parameters_using_expectation_maximisation(
    blocking_rule=block_on("fname", "address[1]"),
    estimate_without_term_frequencies=True,
)

ts4 = linker.training.estimate_parameters_using_expectation_maximisation(
    blocking_rule=block_on("sname", "sa4[1]"),
    estimate_without_term_frequencies=True,
)

In [ ]:
ts1.match_weights_interactive_history_chart()

In [ ]:
linker.visualisations.parameter_estimate_comparisons_chart()

In [ ]:
linker.visualisations.match_weights_chart()

In [ ]:
linker.misc.save_model_to_json("./model.json", overwrite=True)

In [ ]:
df_predict = linker.inference.predict()

In [ ]:
con.sql(
    f"""
    select *
    from {df_predict.physical_name}
    where match_weight > 10 and person_id_l != person_id_r
    """
)

In [ ]:
con.sql(
    f"""
    select 
        person_id_l = person_id_r as true_match
    from {df_predict.physical_name}
    where match_weight > 5
    group by all
    """
)